In [3]:
import numpy as np
file = '/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet/annotations/test.txt'
data = np.loadtxt(file, dtype=str)
len(data)

3669

In [ ]:
"""
Q: What im I looking for: N * dk
k: what I can offer: N * dk
V: What I actually offer: N *dv
"""

'\nQ: What im I looking for: N * dk\nk: what I can offer: N * dk\nV: What I actually offer: N *dk\n'

In [84]:
import numpy as np
import torch
import math
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import math
import torch
import torch.nn as nn


def scaled_dot_product(q, k, v, mask=None):
    d_k = q.shape[-1]
    # 1) typo: matmult -> matmul
    scaled = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # (B,H,Tq,Tk)

    # 2) mask check & application
    if mask is not None:  # don't use `if not mask` on a tensor
        # expect a boolean mask (broadcastable to scaled) where True means "mask"
        # using masked_fill instead of advanced indexing
        if mask.dim() == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)  # -> (1,1,Tq,Tk)
        scaled = scaled.masked_fill(mask, float("-inf"))

    attention = torch.softmax(scaled, dim=-1)

    # 3) matmul usage: attention @ v (not elementwise)
    new_v = torch.matmul(attention, v)  # (B,H,Tq,Dh)
    return new_v


class MultiheadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_dim = 512
        self.num_heads = 8
        self.d_model = 512
        self.head_dim = self.d_model // self.num_heads
        self.qkv_linear = nn.Linear(self.input_dim, 3 * self.d_model)
        self.out_linear = nn.Linear(self.d_model, self.input_dim)

    def forward(self, x, mask=False):
        batch_size, sequence_length, _ = x.shape
        qkv = self.qkv_linear(x)  # (B,T,3*D)
        qkv = qkv.reshape(batch_size, sequence_length, self.num_heads, 3 * self.head_dim)
        qkv = qkv.permute(0, 2, 1, 3)  # (B,H,T,3*Dh)
        q, k, v = qkv.chunk(3, dim=-1)  # each (B,H,T,Dh)
        if mask: #! create mask 
            mask = torch.full((sequence_length, sequence_length), float('-inf'))
            mask = torch.triu(mask, diagonal=1).bool()
        else: mask = None
        new_v = scaled_dot_product(q, k, v, mask)  # (B,H,T,Dh)
        new_v = new_v.reshape(batch_size, sequence_length, self.num_heads * self.head_dim)  # (B,T,D)
        out = self.out_linear(new_v)
        return out


class PositionalEncoding(nn.Module):
    def __init__(self, max_sequence_length = 512, d_model = 512):
        super().__init__()
        self.max_sequence_length = max_sequence_length
        self.d_model = d_model

    def forward(self):
        even_i = torch.arange(0, self.d_model, 2)
        # 4) ensure float exponent math
        denominator = torch.pow(10000.0, (even_i.float() / self.d_model))
        positions = torch.arange(0, self.max_sequence_length).unsqueeze(1)  # (T,1)
        even_PE = torch.sin(positions / denominator)  # (T,D/2)
        odd_PE = torch.cos(positions / denominator)   # (T,D/2)
        positional_encoding = torch.stack((even_PE, odd_PE), dim=-1).reshape(
            self.max_sequence_length, self.d_model
        )
        return positional_encoding  # (T,D)


class LayerNormalization(nn.Module):
    def __init__(self, parameters_shape, eps=1e-5):
        # 5) missing super init
        super().__init__()
        self.parameters_shape = parameters_shape
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(parameters_shape))
        self.beta = nn.Parameter(torch.zeros(parameters_shape))

    def forward(self, input):
        dims = [-(i + 1) for i in range(len(self.parameters_shape))]
        mean = input.mean(dim=dims, keepdim=True)
        var = ((input - mean) ** 2).mean(dim=dims, keepdim=True)
        std = (var + self.eps).sqrt()
        y = (input - mean) / std
        out = self.gamma * y + self.beta
        # 6) return the output (was missing)
        return out

class Encoder(nn.Module):
    def __init__(self, sequence_length, input_dim):
        super().__init__()
        self.sequence_length = sequence_length
        self.attention = MultiheadAttention()
        self.positional_encoding = PositionalEncoding(sequence_length, input_dim)
        self.layer_norm1 = LayerNormalization((input_dim,))
        self.layer_norm2 = LayerNormalization((input_dim, ))
        self.ffn = nn.Sequential( #! 512 -> 1024 -> 512
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(2048, input_dim)
        )
    def forward(self, x):
        # the input vectors are (sequence_length, input_dim)
        after_pe = x + self.positional_encoding[:self.sequence_length, :]
        scaled_attention = self.attention(after_pe, True) #! this is mask true
        norm1 = self.layer_norm1(scaled_attention + after_pe)

        ffn_out = self.ffn(norm1)
        norm2 = self.layer_norm2(ffn_out + norm1) 
        return norm2



In [106]:
sequence_length = 512
batch_size = 2
input_dim = 512

x = torch.randn((batch_size, sequence_length, input_dim))
attention = MultiheadAttention()
print("values_output shape: ", attention(x).shape, "input shape", x.shape)
values_output = attention(x)

pe = PositionalEncoding(sequence_length, input_dim)
positional_encoding = pe()
after_pe = x + positional_encoding[:sequence_length, :]  # (B,T,D) + (T,D) -> broadcast

layer_norm = LayerNormalization((input_dim,))
norm_pe = layer_norm(after_pe)
norm_pe.shape

values_output shape:  torch.Size([2, 512, 512]) input shape torch.Size([2, 512, 512])


torch.Size([2, 512, 512])

In [102]:
class Encoder(nn.Module):
    def __init__(self, sequence_length, input_dim):
        super().__init__()
        self.attention = MultiheadAttention()
        self.positional_encoding = PositionalEncoding(sequence_length, input_dim)
        self.layer_norm1 = LayerNormalization((input_dim,))
        self.layer_norm2 = LayerNormalization((input_dim, ))
        self.ffn = nn.Sequential()
    def forward(self, x):
        # the input vectors are (sequence_length, input_dim)
        after_pe = x + positional_encoding[:sequence_length, :]
        scaled_attention = self.attention(after_pe, True) #! this is mask true
        norm1 = self.layer_norm1(scaled_attention + after_pe)

        ffn_out = self.ffn(norm1)
        norm2 = self.layer_norm2(ffn_out + norm1) 
        return norm2


In [104]:
encoder = Encoder(sequence_length, input_dim)
encoder(x).shape

torch.Size([2, 512, 512])

In [89]:
norm_pe.shape

torch.Size([2, 512, 512])

In [ ]:
norm_pe[0].mean(), norm_pe[0].std()

(tensor(4.0745e-10, grad_fn=<MeanBackward0>),
 tensor(1.0000, grad_fn=<StdBackward0>))

In [50]:
sequence_length = 4
batch_size = 1
input_dim = 512
d_model = 512
num_heads = 8
x = torch.randn(batch_size, sequence_length, input_dim)


In [51]:
qkv_layer = torch.nn.Linear(input_dim, 3 * d_model)
qkv = qkv_layer(x)

In [55]:
head_dim = d_model // num_heads
qkv = qkv.reshape(batch_size, sequence_length, num_heads, 3 * head_dim)
print(qkv.shape)
qkv = qkv.permute(0, 2, 1, 3)
print(qkv.shape)

torch.Size([1, 4, 8, 192])
torch.Size([1, 8, 4, 192])


In [56]:
q, k, v = qkv.chunk(3, dim=-1)
print(q.shape, k.shape, v.shape)

torch.Size([1, 8, 4, 64]) torch.Size([1, 8, 4, 64]) torch.Size([1, 8, 4, 64])


In [ ]:
# mask = torch.full(scaled.shape, float('-inf'))
# mask = torch.triu(mask, diagonal=1)
# mask[0][1]

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

In [58]:
d_k = q.shape[-1]
scaled = torch.matmul(q, k.transpose(-2, -1)) /math.sqrt(d_k) #! this is the normalised softmax shist
inditi = torch.tril(torch.ones((sequence_length, sequence_length))).to(torch.bool)  # Lower triangular matrix
print(scaled.shape, inditi.shape)

torch.Size([1, 8, 4, 4]) torch.Size([4, 4])


In [67]:
scaled[:, :, ~inditi] = float('-inf')
attention = torch.nn.functional.softmax(scaled, dim = -1)
new_v = torch.matmul(attention, v)

In [69]:
new_v.shape

torch.Size([1, 8, 4, 64])

In [71]:
new_v = new_v.reshape(batch_size, sequence_length, num_heads * head_dim)
print(new_v.shape)

torch.Size([1, 4, 512])


In [ ]:
def scaled_dot_product(q, k, v, mask = None):
    d_k = q.shape[-1]
    scaled = torch.matmult(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if not mask:
        scaled[:, :, ~mask] = float('-inf')
    attention = torch.nn.functional.softmax(scaled, dim = -1)
    new_v = torch.matmul(attention) * v
    return new_v

In [72]:
max_sequence_length = 10
d_model = 6 # embedding dimension


"""
P(p, i )  = sin(p / 10000^(2i/d_model)) if i is even
P(p, i )  = cos(p / 10000^(2i/d_model)) if i is odd
"""

even_i = torch.arange(0, d_model, 2)
odd_i = torch.arange(1, d_model, 2)

print(even_i, odd_i)

tensor([0, 2, 4]) tensor([1, 3, 5])


In [78]:
positional_encoding = torch.zeros((max_sequence_length//2, d_model))
positional_encoding   

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])

In [76]:
even_denominator = torch.pow(10000, (even_i/d_model))
even_denominator

tensor([  1.0000,  21.5443, 464.1590])

In [82]:
positional_encoding = torch.sin(even_i.unsqueeze(1) / even_denominator)
positional_encoding

tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.9093,  0.0927,  0.0043],
        [-0.7568,  0.1846,  0.0086]])

In [ ]:
import math
import torch
import torch.nn as nn


def scaled_dot_product(q, k, v, mask=None):
    d_k = q.shape[-1]
    scaled = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # (B,H,Tq,Tk)

    if mask is not None:
        if mask.dim() == 2:  # (T,T) causal mask
            mask = mask.unsqueeze(0).unsqueeze(0)  # -> (1,1,T,T)
        scaled = scaled.masked_fill(mask, float("-inf"))

    attention = torch.softmax(scaled, dim=-1)
    new_v = torch.matmul(attention, v)  # (B,H,Tq,Dh)
    return new_v


class MultiheadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.input_dim = d_model
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = self.d_model // self.num_heads
        # make it 3 times, each for q, k, v
        self.qkv_linear = nn.Linear(self.input_dim, 3 * self.d_model)
        self.out_linear = nn.Linear(self.d_model, self.input_dim)

    def forward(self, x, mask=False):
        batch_size, sequence_length, _ = x.shape
        qkv = self.qkv_linear(x)  # (B,T,3*D)
        qkv = qkv.reshape(batch_size, sequence_length, self.num_heads, 3 * self.head_dim)
        qkv = qkv.permute(0, 2, 1, 3)  # (B,H,T,3*Dh)
        q, k, v = qkv.chunk(3, dim=-1)  # each (B,H,T,Dh)

        if mask:
            # put mask on the same device as x
            m = torch.triu(
                torch.full((sequence_length, sequence_length), float("-inf"), device=x.device),
                diagonal=1,
            ).bool()  # (T,T) True = masked
        else:
            m = None

        # we compute the attention for each of the head indipendently and the combine them  bs, heads, sq, d_h -> bs, sq, d_h * heads
        new_v = scaled_dot_product(q, k, v, m)  # (B,H,T,Dh)

        new_v = new_v.reshape(batch_size, sequence_length, self.num_heads * self.head_dim)  # (B,T,D)
        out = self.out_linear(new_v)
        return out


class PositionalEncoding(nn.Module):
    def __init__(self, max_sequence_length=512, d_model=512):
        super().__init__()
        self.max_sequence_length = max_sequence_length
        self.d_model = d_model

    def forward(self):
        even_i = torch.arange(0, self.d_model, 2)
        denominator = torch.pow(10000.0, (even_i.float() / self.d_model))
        positions = torch.arange(0, self.max_sequence_length).unsqueeze(1)  # (T,1)
        even_PE = torch.sin(positions / denominator)  # (T,D/2)
        odd_PE = torch.cos(positions / denominator)   # (T,D/2)
        positional_encoding = torch.stack((even_PE, odd_PE), dim=-1).reshape(
            self.max_sequence_length, self.d_model
        )
        return positional_encoding  # (T,D)


class LayerNormalization(nn.Module):
    def __init__(self, parameters_shape, eps=1e-5):
        super().__init__()
        self.parameters_shape = parameters_shape
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(parameters_shape))
        self.beta = nn.Parameter(torch.zeros(parameters_shape))

    def forward(self, input):
        dims = [-(i + 1) for i in range(len(self.parameters_shape))]  # e.g. [-1]
        mean = input.mean(dim=dims, keepdim=True)
        var = ((input - mean) ** 2).mean(dim=dims, keepdim=True)
        std = (var + self.eps).sqrt()
        y = (input - mean) / std
        out = self.gamma * y + self.beta
        return out


class Encoder(nn.Module):
    def __init__(self, d_model, ffn_hidden, num_heads, drop_prob, max_sequence_length, num_layers):
        super().__init__()
        self.sequence_length = max_sequence_length
        self.attention = MultiheadAttention(d_model, num_heads)
        # FIX: use max_sequence_length (the arg), not an undefined name
        self.positional_encoding = PositionalEncoding(max_sequence_length, d_model)
        self.layer_norm1 = LayerNormalization((d_model,))
        self.layer_norm2 = LayerNormalization((d_model,))
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden // 2),
            nn.GELU(),
            nn.Dropout(drop_prob),
            nn.Linear(ffn_hidden // 2, ffn_hidden),
            nn.GELU(),
            nn.Dropout(drop_prob),
            nn.Linear(ffn_hidden, d_model),
        )
        self.num_layers = num_layers
        self.dropout1 = nn.Dropout(drop_prob)
        self.dropout2 = nn.Dropout(drop_prob)

    def forward(self, x):
        # x: (B, T, D)
        B, T, D = x.shape
        # FIX: generate PE tensor, move to x's device/dtype, and broadcast over batch
        pe = self.positional_encoding().to(device=x.device, dtype=x.dtype)[:T, :]  # (T,D)
        # print("pe shape: ", pe.shape)
        after_pe = x + pe.unsqueeze(0)  # (B,T,D)
        # for i in range(self.num_layers):
        scaled_attention = self.attention(after_pe, True)  # causal mask
        norm1 = self.layer_norm1(scaled_attention + after_pe)

        ffn_out = self.ffn(norm1)
        norm2 = self.layer_norm2(ffn_out + norm1)
            # after_pe = norm2
            # print("Iteration ", i)
        # print("fj")
        return norm2

class SequentialEncoder(nn.Module):
    def __init__(self, d_model, ffn_hidden, num_heads, drop_prob, max_sequence_length, num_layers):
        super().__init__()
        self.layer = nn.Sequential(*[Encoder(d_model, ffn_hidden, num_heads, drop_prob, max_sequence_length, num_layers) for _ in range(num_layers)])
    def forward(self, x):
        # print(self.layer)
        x = self.layer(x)
        return x


In [15]:
d_model = 512
num_heads = 8
drop_prob = 0.1
max_sequence_length = 200
ffn_hidden = 2048
num_layers = 5
batch_size = 30

input_data = torch.randn((batch_size, max_sequence_length, d_model))
encoder = SequentialEncoder(d_model, ffn_hidden, num_heads, drop_prob, max_sequence_length, num_layers)
encoder = encoder.to('cuda')
encoder(input_data.to('cuda')).shape

torch.Size([30, 200, 512])

In [6]:
d_model = 512
num_heads = 8
drop_prob = 0.1
max_sequence_length = 200
ffn_hidden = 2048
num_layers = 5
batch_size = 30

input_data = torch.randn((batch_size, max_sequence_length, d_model))
encoder = SequentialEncoder(d_model, ffn_hidden, num_heads, drop_prob, max_sequence_length, num_layers)
encoder = encoder
encoder(input_data).shape

torch.Size([30, 200, 512])